In [45]:
from google.cloud import storage
import os
from dotenv import load_dotenv
import pandas as pd
from io import BytesIO

load_dotenv()

True

In [46]:
GCS_BUCKET = os.getenv("GCS_BUCKET", "").strip()
GCS_MEDAL_FETCH = os.getenv("GCS_MEDAL_FETCH", "").strip()
SEASON = int(os.getenv("SEASON"))
BLOB_PATH = f"{GCS_MEDAL_FETCH}/season={SEASON}/laps/all_session_laps.parquet"

BLOB_PATH_SESSIONS = f"gold/season={SEASON}/session.parquet"

In [47]:
client = storage.Client()
bucket = client.bucket(GCS_BUCKET)
blob = bucket.blob(BLOB_PATH)
df = pd.read_parquet(BytesIO(blob.download_as_bytes()))

blob_sessions = bucket.blob(BLOB_PATH_SESSIONS)
df_sessions = pd.read_parquet(BytesIO(blob_sessions.download_as_bytes()))

In [48]:
df.isna().sum()

meeting_key             0
session_key             0
driver_number           0
lap_number              0
date_start             57
duration_sector_1    1846
duration_sector_2     224
duration_sector_3     559
i1_speed             1961
i2_speed              202
is_pit_out_lap          0
lap_duration          629
segments_sector_1      86
segments_sector_2      65
segments_sector_3      84
st_speed              701
ingested_at             0
dtype: int64

In [49]:
keys = ["session_key", "driver_number", "lap_number"]
df.duplicated(subset=keys).sum()

np.int64(0)

**MEETING KEY**

In [54]:
session_start = (
    df_sessions.drop_duplicates("session_key").set_index("session_key")["date_start"]
)

df["date_start"] = df["date_start"].fillna(df["session_key"].map(session_start))
df["date_start"] = pd.to_datetime(df["date_start"], errors="coerce")
df.isna().sum()

meeting_key             0
session_key             0
driver_number           0
lap_number              0
date_start              0
duration_sector_1    1846
duration_sector_2     224
duration_sector_3     559
i1_speed             1961
i2_speed              202
is_pit_out_lap          0
lap_duration          629
segments_sector_1      86
segments_sector_2      65
segments_sector_3      84
st_speed              701
ingested_at             0
dtype: int64

In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18031 entries, 0 to 18030
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   meeting_key        18031 non-null  int64              
 1   session_key        18031 non-null  int64              
 2   driver_number      18031 non-null  int64              
 3   lap_number         18031 non-null  int64              
 4   date_start         18031 non-null  datetime64[ns, UTC]
 5   duration_sector_1  16185 non-null  float64            
 6   duration_sector_2  17807 non-null  float64            
 7   duration_sector_3  17472 non-null  float64            
 8   i1_speed           16070 non-null  float64            
 9   i2_speed           17829 non-null  float64            
 10  is_pit_out_lap     18031 non-null  bool               
 11  lap_duration       17402 non-null  float64            
 12  segments_sector_1  17945 non-null  object     

In [56]:
df.head()

,meeting_key,session_key,driver_number,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,is_pit_out_lap,lap_duration,segments_sector_1,segments_sector_2,segments_sector_3,st_speed,ingested_at
0,1304,11465,31,1,2026-02-11 07:00:00+00:00,NaN,NaN,24.319,NaN,NaN,False,99.848,None,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",NaN,2026-05-04 17:40:16.262855+00:00
1,1304,11465,5,1,2026-02-11 07:00:00+00:00,32.258,44.634,27.954,220.0,237.0,False,104.846,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[2049.0, 2049.0, 2048.0, 2048.0, 2049.0, 2049....","[2048.0, 2048.0, 2048.0, 2048.0, 2048.0, 2048....",NaN,2026-05-04 17:40:16.262855+00:00
2,1304,11465,41,1,2026-02-11 07:00:00+00:00,32.406,45.046,24.557,224.0,224.0,False,102.009,"[nan, nan, nan, nan, nan, nan, nan, 2048.0, 20...","[2048.0, 2049.0, 2048.0, 2048.0, 2048.0, 2048....","[2048.0, 2048.0, 2048.0, 2048.0, 2048.0, 2048....",NaN,2026-05-04 17:40:16.262855+00:00
3,1304,11465,55,1,2026-02-11 07:00:00+00:00,NaN,54.802,28.621,178.0,182.0,True,392.014,"[nan, 2064.0, 2064.0, 2048.0, 2048.0, 2048.0, ...","[2048.0, 2048.0, 2048.0, 2048.0, 2048.0, 2048....","[2048.0, 2048.0, 2048.0, 2048.0, 2048.0, 2048....",178.0,2026-05-04 17:40:16.262855+00:00
4,1304,11465,81,1,2026-02-11 07:00:00+00:00,NaN,NaN,NaN,NaN,NaN,False,NaN,None,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",NaN,2026-05-04 17:40:16.262855+00:00
